# 🚗 MMDetection3D Benchmark & Reference R40 Evaluation Pipeline (Colab & Local)

Notebook này dùng để chạy suy luận (inference) và **đánh giá độc lập 5 mô hình 3D LiDAR Object Detection kinh điển** từ framework OpenMMLab MMDetection3D:
1. **PointPillars** (Single-stage, Pillar-based 2D CNN encoder)
2. **SECOND** (Single-stage, Voxel-based 3D Sparse Convolution)
3. **PointRCNN** (Two-stage, Point-based 3D detector trên raw points)
4. **Part-A²** (Two-stage, Voxel/Point-based với Part-awareness & Part-aggregation)
5. **PV-RCNN** (Two-stage, Hybrid Point-Voxel Set Abstraction hàng đầu)

---

### 🎯 Điểm cốt lõi về độ chuẩn xác & tính công bằng:
* **Sử dụng đúng Evaluator R40 của codebase**: Toàn bộ dự đoán của 5 mô hình MMDetection3D được xuất ra định dạng chuẩn KITTI (`.txt`) và đánh giá thông qua module [`tools.kitti_training_pipeline.evaluate_kitti_3d`](file:///home/duyennh/AI_projects/research_lidar/Lidar/Code/tools/kitti_training_pipeline/evaluate_kitti_3d.py) với hạt nhân GPU [`kitti_reference/eval.py`](file:///home/duyennh/AI_projects/research_lidar/Lidar/Code/tools/kitti_training_pipeline/kitti_reference/eval.py) được ghim mã băm SHA256 chính thức từ OpenPCDet.
* **Đồng nhất tập dữ liệu**: Đánh giá trên đúng tập validation `splits/kitti/val.txt` (3.769 frames) của KITTI với Ground Truth nguyên bản `training/label_2`.
* **Đầy đủ toàn diện chỉ số (Full Metrics)**:
  - **Tốc độ & Độ trễ**: Đo đạc chính xác **Latency (ms/frame)**, **FPS (Frames/sec)** và tổng thời gian thực thi.
  - **Độ chính xác mAP**: Xuất đầy đủ **mAP Easy (%)**, **mAP Moderate (%)** (chỉ số xếp hạng chính thức), **mAP Hard (%)**, và **mAP9 (%)** (trung bình 9 ô).
  - **Chi tiết từng lớp**: Điểm AP chi tiết của cả 3 lớp (*Car*, *Pedestrian*, *Cyclist*) theo cả 2 chế độ **3D AP** và **BEV AP**.
* **Xuất bảng báo cáo công bố**: Tự động xuất ra các file CSV hoàn chỉnh (`benchmark_3D_mmdet3d_full.csv`, `benchmark_BEV_mmdet3d_full.csv`, `benchmark_summary_mmdet3d.csv`) và báo cáo Markdown/LaTeX chuẩn mực cho bài báo khoa học.

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Mounting & Dependency Verification
# ==============================================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path
from shlex import quote as q

# 1. Nhận diện môi trường thực thi (Google Colab vs Local Workstation)
IN_COLAB = "google.colab" in sys.modules
GIT_REPO_URL = "https://github.com/danhyoyo/Lidar.git"
GIT_BRANCH = "research/mobilepixornext-under1m"

def find_code_root(candidate):
    candidate = Path(candidate).expanduser().resolve()
    for root in (candidate / "Code", candidate):
        if ((root / "configs/config.json").is_file()
                and (root / "detector").is_dir()
                and (root / "tools/kitti_training_pipeline").is_dir()):
            return root
    return None

if IN_COLAB:
    print("Detected Google Colab environment. Mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_CHECKOUT_DIR = Path("/content/Lidar").resolve()
    DRIVE_BASE = Path("/content/drive/MyDrive/lidar_project").resolve()

    candidates = [DEFAULT_CHECKOUT_DIR, Path.cwd(), Path("/content/drive/MyDrive/Lidar")]
    DEFAULT_REPO_DIR = next((root for c in candidates if (root := find_code_root(c)) is not None), None)

    if DEFAULT_REPO_DIR is None:
        print(f"Cloning codebase branch '{GIT_BRANCH}' into {DEFAULT_CHECKOUT_DIR}...")
        subprocess.run(["git", "clone", "-b", GIT_BRANCH, GIT_REPO_URL, str(DEFAULT_CHECKOUT_DIR)], check=True)
        DEFAULT_REPO_DIR = find_code_root(DEFAULT_CHECKOUT_DIR)
        if DEFAULT_REPO_DIR is None:
            raise FileNotFoundError(f"Cannot find Code root in {DEFAULT_CHECKOUT_DIR}")
        print(f"✓ Cloned branch '{GIT_BRANCH}' successfully.")
    
    print("Installing OpenMMLab & MMDetection3D requirements...")
    !pip install -q -U openmim
    !mim install -q mmengine "mmcv>=2.0.0" "mmdet>=3.0.0" "mmdet3d>=1.1.0"
    
    # Cài đặt spconv phù hợp với phiên bản CUDA
    import torch
    cuda_ver = torch.version.cuda
    if cuda_ver:
        cu_tag = f"cu{cuda_ver.replace('.', '')[:3]}"
        print(f"Installing spconv for CUDA {cuda_ver} ({cu_tag})...")
        !pip install -q spconv-{cu_tag} || pip install -q spconv-cu120 || pip install -q spconv-cu118
        pass
    
    # Cài đặt requirements cho evaluator reference của codebase
    if (DEFAULT_REPO_DIR / "requirements-kitti.txt").is_file():
        !pip install -q -r {q(str(DEFAULT_REPO_DIR / "requirements-kitti.txt"))}
        pass
    !pip install -q pandas tabulate

    DEFAULT_ARTIFACT_ROOT = DRIVE_BASE / "artifacts_benchmark_mmdet3d"
    DEFAULT_DRIVE_RAW_DIR = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP").resolve()
    DEFAULT_RAW_ROOT = Path("/content/kitti_raw").resolve()
    MMDET3D_ROOT = Path("/content/mmdetection3d").resolve()
else:
    print("Running in local workstation environment.")
    local_candidates = [Path.cwd(), *Path.cwd().parents]
    DEFAULT_REPO_DIR = next((root for c in local_candidates if (root := find_code_root(c)) is not None), None)
    if DEFAULT_REPO_DIR is None:
        raise FileNotFoundError("No project code directory found. Please run from the Lidar repository.")
    DEFAULT_ARTIFACT_ROOT = (DEFAULT_REPO_DIR / "artifacts_benchmark_mmdet3d").resolve()
    DEFAULT_DRIVE_RAW_DIR = (DEFAULT_REPO_DIR / "data/kitti/raw").resolve()
    DEFAULT_RAW_ROOT = (DEFAULT_REPO_DIR / "data/kitti/raw").resolve()
    MMDET3D_ROOT = (DEFAULT_REPO_DIR.parent / "mmdetection3d").resolve()

# 2. Thiết lập cấu hình thư mục cơ sở
REPO_DIR = DEFAULT_REPO_DIR
ARTIFACT_ROOT = DEFAULT_ARTIFACT_ROOT
DRIVE_RAW_DIR = DEFAULT_DRIVE_RAW_DIR
RAW_KITTI_ROOT = DEFAULT_RAW_ROOT

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

# 3. Clone repository mmdetection3d để truy xuất configs & tools chuẩn nếu chưa có
if not (MMDET3D_ROOT / "configs").is_dir():
    print(f"Cloning official OpenMMLab MMDetection3D to {MMDET3D_ROOT}...")
    subprocess.run(["git", "clone", "-b", "main", "https://github.com/open-mmlab/mmdetection3d.git", str(MMDET3D_ROOT)], check=True)

# 4. Thêm các đường dẫn cần thiết vào sys.path
pipeline_dir = REPO_DIR / "tools/kitti_training_pipeline"
source_paths = [str(pipeline_dir), str(REPO_DIR / "detector"), str(REPO_DIR), str(MMDET3D_ROOT)]
for p in source_paths:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 80)
print(f"Working Code Directory  : {REPO_DIR}")
print(f"MMDetection3D Root      : {MMDET3D_ROOT}")
print(f"Artifacts Destination   : {ARTIFACT_ROOT}")
print(f"Local Raw KITTI Dir     : {RAW_KITTI_ROOT}")
print(f"Compute Device          : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU Hardware            : {torch.cuda.get_device_name(0)}")
    from numba import cuda
    print(f"Numba CUDA Available    : {cuda.is_available()} (Required for R40 Reference Evaluator)")
print("=" * 80)
print("✓ Environment setup completed.")

In [ ]:
# ==============================================================================
# Cell 2: Quick Configuration Control Panel
# ==============================================================================
from pathlib import Path

# --- 1. LỰA CHỌN MÔ HÌNH ĐÁNH GIÁ ---
# Có thể chọn bất kỳ danh sách nào trong 5 mô hình:
# ["PointPillars", "SECOND", "PointRCNN", "Part-A2", "PV-RCNN"]
ACTIVE_MODELS = [
    "PointPillars",
    "SECOND",
    "PointRCNN",
    "Part-A2",
    "PV-RCNN",
]

# --- 2. CẤU HÌNH SUY LUẬN & ĐÁNH GIÁ ---
BATCH_SIZE = 4                # Kích thước batch khi nạp dữ liệu kiểm tra (giảm xuống 2 nếu VRAM < 8GB)
NUM_WORKERS = 4               # Số tiến trình nạp dữ liệu
VAL_SPLIT = "splits/kitti/val.txt"  # Tập validation chuẩn (3.769 frames)
MAX_FRAMES = None             # None: chạy toàn bộ 3.769 frames; hoặc đặt số int (vd: 50) để test nhanh

# Thư mục lưu kết quả benchmark
BENCHMARK_RUN_NAME = "mmdet3d_5models_kitti_val_r40"
BENCHMARK_DIR = ARTIFACT_ROOT / BENCHMARK_RUN_NAME
PREDICTIONS_DIR = BENCHMARK_DIR / "predictions"
EVAL_RESULTS_DIR = BENCHMARK_DIR / "evaluation_r40"

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("🎯 BENCHMARK CONFIGURATION:")
print(f"Selected Models : {ACTIVE_MODELS}")
print(f"Validation Split: {REPO_DIR / VAL_SPLIT}")
print(f"Max Frames      : {'ALL (3,769 frames)' if MAX_FRAMES is None else MAX_FRAMES}")
print(f"Output Directory: {BENCHMARK_DIR}")
print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 3: Extract KITTI Dataset (.rar / .zip) from Google Drive to Local Colab SSD
# ==============================================================================
import os
import sys
import shutil
from pathlib import Path
from shlex import quote as q

# Đảm bảo tiện ích unrar đã sẵn sàng
if shutil.which("unrar") is None:
    print("Installing unrar utility for extracting .rar files...")
    !apt-get update -qq && apt-get install -y -qq unrar
    pass

DRIVE_DATASET_PATH = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP").resolve() if IN_COLAB else DRIVE_RAW_DIR

def auto_detect_kitti_drive_path(current_path: Path) -> Path:
    def has_kitti_archives(folder: Path) -> bool:
        if not folder.is_dir():
            return False
        files = [p.name.lower() for p in folder.iterdir() if p.is_file()]
        return any((ext in f) and any(k in f for k in ("velodyne", "label", "calib", "kitti"))
                   for f in files for ext in (".rar", ".zip", ".tar"))

    if has_kitti_archives(current_path):
        return current_path

    search_dirs = [
        Path("/content/drive/MyDrive/KITTI_DATASET_ZIP"),
        Path("/content/drive/MyDrive/KITTI_ZIP"),
        Path("/content/drive/MyDrive/KITTI_RAW"),
        Path("/content/drive/MyDrive/kitti_raw"),
        Path("/content/drive/MyDrive/kitti"),
        Path("/content/drive/MyDrive/KITTI"),
        Path("/content/drive/MyDrive/KITTI_DATASET"),
        Path("/content/drive/MyDrive/data_object"),
        Path("/content/drive/MyDrive/lidar_project/raw"),
        Path("/content/drive/MyDrive"),
    ]
    for d in search_dirs:
        if has_kitti_archives(d):
            print(f"💡 Tự động phát hiện các file nén KITTI tại: {d}")
            return d.resolve()
    return current_path

DRIVE_DATASET_PATH = auto_detect_kitti_drive_path(DRIVE_DATASET_PATH)

RAW_TRAINING = RAW_KITTI_ROOT if RAW_KITTI_ROOT.name == "training" else RAW_KITTI_ROOT / "training"
EXTRACTION_ROOT = RAW_TRAINING.parent
RAW_TRAINING.mkdir(parents=True, exist_ok=True)
EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)

archives_spec = {
    "velodyne": ("*.bin", 7481, ["velodyne"]),
    "label_2": ("*.txt", 7481, ["label_2", "label", "labels"]),
    "calib": ("*.txt", 7481, ["calib"]),
    "image_2": ("*.png", 7481, ["image_2", "image", "images"]),
}

print(f"Checking dataset files in local SSD: {RAW_TRAINING}...")

for name, (pattern, expected_count, keywords) in archives_spec.items():
    target_folder = RAW_TRAINING / name
    current_count = len(list(target_folder.glob(pattern))) if target_folder.is_dir() else 0
    if current_count >= expected_count:
        print(f"[{name.upper()}] Sẵn sàng: tìm thấy {current_count}/{expected_count} files.")
        continue

    print(f"[{name.upper()}] Chưa đủ files ({current_count}/{expected_count}). Tìm kiếm trên Drive...")
    archive_path = None
    if DRIVE_DATASET_PATH.is_dir():
        for ext in ("*.rar", "*.zip", "*.tar", "*.tar.gz"):
            for candidate in sorted(DRIVE_DATASET_PATH.rglob(ext)):
                cand_lower = candidate.name.lower()
                if any(kw in cand_lower for kw in keywords):
                    archive_path = candidate
                    break
            if archive_path is not None:
                break

    if archive_path is not None:
        print(f"Giải nén {archive_path.name} -> {EXTRACTION_ROOT}...")
        ext = archive_path.suffix.lower()
        if ext == ".rar":
            !unrar x -o+ {q(str(archive_path))} {q(str(EXTRACTION_ROOT))}/
            pass
        elif ext in (".tar", ".gz"):
            !tar -xf {q(str(archive_path))} -C {q(str(EXTRACTION_ROOT))}
            pass
        else:
            !unzip -q -o {q(str(archive_path))} -d {q(str(EXTRACTION_ROOT))}
            pass

        # Tổ chức lại nếu thư mục bị lồng nhau
        if len(list(target_folder.glob(pattern))) < expected_count:
            for found_dir in EXTRACTION_ROOT.rglob(name):
                if found_dir.is_dir() and found_dir.resolve() != target_folder.resolve():
                    if len(list(found_dir.glob(pattern))) >= expected_count:
                        shutil.rmtree(target_folder, ignore_errors=True)
                        shutil.move(str(found_dir), str(target_folder))
                        break
    else:
        # Nếu đã có sẵn thư mục chưa nén trên Drive, copy sang local SSD
        drive_folder_candidates = [DRIVE_DATASET_PATH / name, DRIVE_DATASET_PATH / "training" / name]
        for kw in keywords:
            drive_folder_candidates.extend([DRIVE_DATASET_PATH / kw, DRIVE_DATASET_PATH / "training" / kw])
        drive_folder = next((p for p in drive_folder_candidates if p.is_dir()), None)
        if drive_folder is not None:
            print(f"Copying {name} from Drive: {drive_folder}...")
            !cp -r {q(str(drive_folder))} {q(str(target_folder))}
            pass
        else:
            if name != "image_2":
                raise FileNotFoundError(f"Không tìm thấy dữ liệu '{name}' trên Drive: {DRIVE_DATASET_PATH}")
            else:
                print("⚠️ Cảnh báo: Thiếu image_2 (ảnh RGB). Một số mô hình LiDAR thuần túy vẫn có thể chạy bình thường.")

    final_count = len(list(target_folder.glob(pattern))) if target_folder.is_dir() else 0
    print(f"[{name.upper()}] Hiện có: {final_count} files trong {target_folder}.")

# 3. Đồng bộ cấu trúc thư mục mà MMDetection3D yêu cầu: data/kitti
MMDET_KITTI_ROOT = MMDET3D_ROOT / "data/kitti"
MMDET_KITTI_ROOT.mkdir(parents=True, exist_ok=True)
(MMDET_KITTI_ROOT / "ImageSets").mkdir(parents=True, exist_ok=True)

# Tạo liên kết symlink hoặc copy dữ liệu sang MMDET_KITTI_ROOT/training
mmdet_training = MMDET_KITTI_ROOT / "training"
if not mmdet_training.exists():
    try:
        os.symlink(RAW_TRAINING, mmdet_training, target_is_directory=True)
        print(f"✓ Created symlink: {mmdet_training} -> {RAW_TRAINING}")
    except OSError:
        shutil.copytree(RAW_TRAINING, mmdet_training)

# Sao chép file split validation sang ImageSets/val.txt (dạng ID chuẩn không chứa nhãn)
from tools.kitti_training_pipeline.prepare_kitti import read_ids
val_split_file = REPO_DIR / VAL_SPLIT
val_ids = read_ids(val_split_file)
if MAX_FRAMES is not None:
    val_ids = val_ids[:MAX_FRAMES]

imagesets_val_txt = MMDET_KITTI_ROOT / "ImageSets/val.txt"
imagesets_val_txt.write_text("\n".join(val_ids) + "\n", encoding="utf-8")
print(f"✓ Saved {len(val_ids):,} evaluation frame IDs to {imagesets_val_txt}")
print("✓ KITTI dataset ready for MMDetection3D.")

In [ ]:
# ==============================================================================
# Cell 4: Fast KITTI Info Preparation (kitti_infos_val.pkl)
# ==============================================================================
import pickle
import numpy as np
from pathlib import Path
from tools.kitti_training_pipeline.evaluate_kitti_3d import read_annotation

INFO_VAL_PKL = MMDET_KITTI_ROOT / "kitti_infos_val.pkl"

def generate_fast_kitti_infos_val(kitti_root: Path, val_ids_list: list, out_path: Path):
    """Tạo nhanh file kitti_infos_val.pkl cho tập validation mà không cần sinh GT database lâu."""
    print(f"Building fast kitti_infos_val.pkl for {len(val_ids_list):,} frames...")
    infos = []
    training_dir = kitti_root / "training"
    calib_dir = training_dir / "calib"
    label_dir = training_dir / "label_2"
    image_dir = training_dir / "image_2"
    velodyne_dir = training_dir / "velodyne"

    for idx_str in val_ids_list:
        idx = int(idx_str)
        info = {
            "sample_idx": idx,
            "image": {"image_idx": idx, "image_path": f"training/image_2/{idx_str}.png", "image_shape": np.array([375, 1242], dtype=np.int32)},
            "point_cloud": {"num_features": 4, "velodyne_path": f"training/velodyne/{idx_str}.bin"},
            "calib": {},
            "annos": {}
        }
        
        # Đọc calib nếu có
        calib_file = calib_dir / f"{idx_str}.txt"
        if calib_file.is_file():
            calib_data = {}
            for line in calib_file.read_text().splitlines():
                if not line.strip(): continue
                key, vals = line.split(":", 1)
                calib_data[key.strip()] = np.array([float(x) for x in vals.strip().split()], dtype=np.float32)
            if "P2" in calib_data:
                info["calib"]["P2"] = calib_data["P2"].reshape(3, 4)
            if "R0_rect" in calib_data:
                info["calib"]["R0_rect"] = calib_data["R0_rect"].reshape(3, 3)
            if "Tr_velo_to_cam" in calib_data:
                info["calib"]["Tr_velo_to_cam"] = calib_data["Tr_velo_to_cam"].reshape(3, 4)

        # Đọc annotation nhãn
        lbl_file = label_dir / f"{idx_str}.txt"
        if lbl_file.is_file():
            anno = read_annotation(lbl_file, prediction=False)
            if len(anno["name"]) > 0:
                info["annos"] = {
                    "name": anno["name"],
                    "truncated": anno["truncated"],
                    "occluded": anno["occluded"],
                    "alpha": anno["alpha"],
                    "bbox": anno["bbox"],
                    "dimensions": anno["dimensions"][:, [1, 2, 0]],  # h, w, l
                    "location": anno["location"],
                    "rotation_y": anno["rotation_y"],
                }
                info["annos"]["index"] = np.arange(len(anno["name"]), dtype=np.int32)
                info["annos"]["group_ids"] = np.arange(len(anno["name"]), dtype=np.int32)
            else:
                info["annos"] = {
                    "name": np.array([], dtype=str),
                    "truncated": np.array([], dtype=np.float32),
                    "occluded": np.array([], dtype=np.int32),
                    "alpha": np.array([], dtype=np.float32),
                    "bbox": np.zeros((0, 4), dtype=np.float32),
                    "dimensions": np.zeros((0, 3), dtype=np.float32),
                    "location": np.zeros((0, 3), dtype=np.float32),
                    "rotation_y": np.array([], dtype=np.float32),
                    "index": np.array([], dtype=np.int32),
                    "group_ids": np.array([], dtype=np.int32),
                }

        infos.append(info)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "wb") as f:
        pickle.dump(infos, f)
    print(f"✓ Created {out_path} ({len(infos):,} validation frames).")

# Kiểm tra nếu chưa có hoặc có lệnh tạo lại
if not INFO_VAL_PKL.is_file():
    generate_fast_kitti_infos_val(MMDET_KITTI_ROOT, val_ids, INFO_VAL_PKL)
else:
    print(f"✓ Found existing: {INFO_VAL_PKL}")

In [ ]:
# ==============================================================================
# Cell 5: Model Zoo Checkpoints Downloader
# ==============================================================================
import urllib.request
from pathlib import Path

CHECKPOINTS_DIR = MMDET3D_ROOT / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Thông tin 5 mô hình chuẩn OpenMMLab MMDetection3D (KITTI 3-class: Car, Pedestrian, Cyclist)
MODELS_CATALOG = {
    "PointPillars": {
        "config": MMDET3D_ROOT / "configs/pointpillars/pointpillars_hv_secfpn_8xb6-160e_kitti-3d-3class.py",
        "url": "https://download.openmmlab.com/mmdetection3d/v1.0.0_models/pointpillars/hv_pointpillars_secfpn_6x8_160e_kitti-3d-3class/hv_pointpillars_secfpn_6x8_160e_kitti-3d-3class_20220301_150306-37dc2420.pth",
        "weight_file": "pointpillars_hv_secfpn_kitti_3class.pth",
        "type": "Pillar-based 2D CNN",
        "params_approx_m": 4.83,
    },
    "SECOND": {
        "config": MMDET3D_ROOT / "configs/second/second_hv_secfpn_8xb6-80e_kitti-3d-3class.py",
        "url": "https://download.openmmlab.com/mmdetection3d/v1.0.0_models/second/hv_second_secfpn_6x8_80e_kitti-3d-3class/hv_second_secfpn_6x8_80e_kitti-3d-3class_20200620_230238-925562e0.pth",
        "weight_file": "second_hv_secfpn_kitti_3class.pth",
        "type": "Voxel Sparse 3D Conv",
        "params_approx_m": 5.33,
    },
    "PointRCNN": {
        "config": MMDET3D_ROOT / "configs/point_rcnn/point-rcnn_8xb2_kitti-3d-3class.py",
        "url": "https://download.openmmlab.com/mmdetection3d/v1.0.0_models/point_rcnn/point_rcnn_2x8_kitti-3d-3classes_20211208_151344.pth",
        "weight_file": "point_rcnn_2x8_kitti_3classes.pth",
        "type": "Point-based Two-Stage",
        "params_approx_m": 4.04,
    },
    "Part-A2": {
        "config": MMDET3D_ROOT / "configs/parta2/hv_PartA2_secfpn_2x8_cyclic_80e_kitti-3d-3class.py",
        "url": "https://download.openmmlab.com/mmdetection3d/v1.0.0_models/parta2/hv_PartA2_secfpn_2x8_cyclic_80e_kitti-3d-3class/hv_PartA2_secfpn_2x8_cyclic_80e_kitti-3d-3class_20201202_174755-9a84a6ba.pth",
        "weight_file": "parta2_secfpn_kitti_3class.pth",
        "type": "Part-Aware Voxel/Point",
        "params_approx_m": 63.81,
    },
    "PV-RCNN": {
        "config": MMDET3D_ROOT / "configs/pv_rcnn/pv_rcnn_8xb2-80e_kitti-3d-3class.py",
        "url": "https://download.openmmlab.com/mmdetection3d/v1.1.0_models/pv_rcnn/pv_rcnn_8xb2-80e_kitti-3d-3class_20230211_025219-c90a1914.pth",
        "weight_file": "pv_rcnn_8xb2_kitti_3class.pth",
        "type": "Point-Voxel Hybrid",
        "params_approx_m": 13.12,
    },
}

def download_file_with_progress(url, dest_path):
    if dest_path.is_file() and dest_path.stat().st_size > 1024 * 1024:
        print(f"✓ Đã có sẵn: {dest_path.name} ({dest_path.stat().st_size / (1024**2):.1f} MB)")
        return
    print(f"Downloading {dest_path.name} from {url}...")
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, dest_path)
    print(f"✓ Downloaded {dest_path.name} ({dest_path.stat().st_size / (1024**2):.1f} MB).")

print("=" * 80)
print("📥 CHECKPOINT DOWNLOAD MANAGER:")
for model_name in ACTIVE_MODELS:
    if model_name not in MODELS_CATALOG:
        print(f"⚠️ Warning: Model '{model_name}' không có trong catalog!")
        continue
    spec = MODELS_CATALOG[model_name]
    target_weight = CHECKPOINTS_DIR / spec["weight_file"]
    spec["local_checkpoint"] = target_weight
    download_file_with_progress(spec["url"], target_weight)

print("=" * 80)
print("✓ All active model checkpoints verified.")

In [ ]:
# ==============================================================================
# Cell 6: Batch Inference & Performance Profiling (FPS, Latency ms & KITTI Export)
# ==============================================================================
import time
import subprocess
from pathlib import Path
from shlex import quote as q

inference_timings = {}

for model_name in ACTIVE_MODELS:
    if model_name not in MODELS_CATALOG:
        continue
    spec = MODELS_CATALOG[model_name]
    cfg_file = spec["config"]
    ckpt_file = spec["local_checkpoint"]

    if not cfg_file.is_file():
        candidates = list(MMDET3D_ROOT.glob(f"configs/**/{cfg_file.name}"))
        if candidates:
            cfg_file = candidates[0]
            spec["config"] = cfg_file
        else:
            print(f"❌ Không tìm thấy file config cho {model_name}: {cfg_file}")
            continue

    model_sub_dir = PREDICTIONS_DIR / model_name
    model_sub_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 80)
    print(f"🚀 INFERENCE & PROFILING: {model_name.upper()}")
    print(f"   - Architecture : {spec['type']} (~{spec['params_approx_m']}M params)")
    print(f"   - Config       : {cfg_file}")
    print(f"   - Checkpoint   : {ckpt_file}")
    print(f"   - Output Dir   : {model_sub_dir}")
    print("=" * 80)

    # Đo đạc thời gian end-to-end Pipeline (DataLoader + Preprocess + Forward + Postprocess + Txt dump)
    start_time = time.time()
    cmd = [
        sys.executable,
        str(MMDET3D_ROOT / "tools/test.py"),
        str(cfg_file),
        str(ckpt_file),
        "--cfg-options",
        f"test_evaluator.submission_prefix={str(model_sub_dir)}",
        f"test_dataloader.batch_size={BATCH_SIZE}",
        f"test_dataloader.num_workers={NUM_WORKERS}",
    ]

    try:
        res = subprocess.run(cmd, cwd=str(MMDET3D_ROOT), check=True, text=True, capture_output=True)
        elapsed = time.time() - start_time
        num_frames = len(val_ids)
        latency_ms = (elapsed / num_frames) * 1000.0 if num_frames > 0 else 0.0
        fps = num_frames / elapsed if elapsed > 0 else 0.0
        
        inference_timings[model_name] = {
            "elapsed_seconds": elapsed,
            "latency_ms": latency_ms,
            "fps": fps,
            "num_frames": num_frames
        }
        print(f"✓ {model_name} profiling completed:")
        print(f"   - Total Time : {elapsed:.2f} seconds ({num_frames:,} frames)")
        print(f"   - Latency    : {latency_ms:.2f} ms / frame")
        print(f"   - Speed      : {fps:.2f} FPS")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error during inference for {model_name}:\n{e.stderr[-1000:]}")
        continue

    # Xác minh các file dự đoán được xuất
    txt_files = list(model_sub_dir.rglob("*.txt"))
    print(f"   - Exported {len(txt_files):,} prediction .txt files in {model_sub_dir}.")

    # Đảm bảo mỗi frame ID trong val_ids đều có 1 file .txt
    pred_dir = next((p.parent for p in txt_files if p.name.endswith(".txt")), model_sub_dir)
    spec["prediction_dir"] = pred_dir
    missing_cnt = 0
    for fid in val_ids:
        fpath = pred_dir / f"{fid}.txt"
        if not fpath.is_file():
            fpath.touch()
            missing_cnt += 1
    if missing_cnt > 0:
        print(f"   - Handled {missing_cnt} frames with 0 detections by creating clean empty files.")

print("=" * 80)
print("✓ Completed prediction exports and timing profiling for all selected models.")

In [ ]:
# ==============================================================================
# Cell 7: Codebase Reference R40 Evaluation Engine (AP3D & APBEV Full Calculation)
# ==============================================================================
import json
from pathlib import Path
from tools.kitti_training_pipeline.evaluate_kitti_3d import evaluate_saved_predictions

evaluation_records = {}

print("=" * 80)
print("📊 RUNNING PINNED CODEBASE REFERENCE R40 EVALUATION ENGINE:")
print(f"Split Path : {REPO_DIR / VAL_SPLIT} ({len(val_ids):,} frames)")
print(f"KITTI Root : {RAW_TRAINING}")
print("=" * 80)

for model_name in ACTIVE_MODELS:
    if model_name not in MODELS_CATALOG:
        continue
    spec = MODELS_CATALOG[model_name]
    pred_dir = spec.get("prediction_dir")
    
    if pred_dir is None or not Path(pred_dir).is_dir():
        model_sub = PREDICTIONS_DIR / model_name
        txts = list(model_sub.rglob("*.txt"))
        if txts:
            pred_dir = txts[0].parent
        else:
            print(f"⚠️ Bỏ qua {model_name}: không tìm thấy file dự đoán.")
            continue

    print(f"\n🔍 Đang đánh giá {model_name}...")
    out_json_3d = EVAL_RESULTS_DIR / f"{model_name}_eval_3d.json"
    out_json_bev = EVAL_RESULTS_DIR / f"{model_name}_eval_bev.json"

    # 1. Đánh giá 3D Object Detection (AP_3D)
    res_3d = evaluate_saved_predictions(
        predictions=pred_dir,
        kitti_root=RAW_TRAINING,
        split_path=REPO_DIR / VAL_SPLIT,
        metric_mode="3d",
        output_path=out_json_3d
    )

    # 2. Đánh giá Bird's Eye View (AP_BEV)
    res_bev = evaluate_saved_predictions(
        predictions=pred_dir,
        kitti_root=RAW_TRAINING,
        split_path=REPO_DIR / VAL_SPLIT,
        metric_mode="bev",
        output_path=out_json_bev
    )

    # Trích xuất đầy đủ mAP Easy, Moderate, Hard, và mAP9 cho 3D
    acc_3d_r40 = res_3d["accuracy"]["R40"]
    car_3d = acc_3d_r40["per_class"]["Car"]
    ped_3d = acc_3d_r40["per_class"]["Pedestrian"]
    cyc_3d = acc_3d_r40["per_class"]["Cyclist"]
    map_easy_3d = (car_3d["Easy"] + ped_3d["Easy"] + cyc_3d["Easy"]) / 3.0
    map_mod_3d = acc_3d_r40["map_moderate_percent"]
    map_hard_3d = (car_3d["Hard"] + ped_3d["Hard"] + cyc_3d["Hard"]) / 3.0
    map9_3d = acc_3d_r40["mean_ap_9_percent"]

    # Trích xuất đầy đủ mAP Easy, Moderate, Hard, và mAP9 cho BEV
    acc_bev_r40 = res_bev["accuracy"]["R40"]
    car_bev = acc_bev_r40["per_class"]["Car"]
    ped_bev = acc_bev_r40["per_class"]["Pedestrian"]
    cyc_bev = acc_bev_r40["per_class"]["Cyclist"]
    map_easy_bev = (car_bev["Easy"] + ped_bev["Easy"] + cyc_bev["Easy"]) / 3.0
    map_mod_bev = acc_bev_r40["map_moderate_percent"]
    map_hard_bev = (car_bev["Hard"] + ped_bev["Hard"] + cyc_bev["Hard"]) / 3.0
    map9_bev = acc_bev_r40["mean_ap_9_percent"]

    timing = inference_timings.get(model_name, {"latency_ms": 0.0, "fps": 0.0, "elapsed_seconds": 0.0})

    evaluation_records[model_name] = {
        "3d": {
            "raw": res_3d,
            "mAP_Easy": map_easy_3d,
            "mAP_Mod": map_mod_3d,
            "mAP_Hard": map_hard_3d,
            "mAP9": map9_3d,
            "Car": car_3d,
            "Pedestrian": ped_3d,
            "Cyclist": cyc_3d,
        },
        "bev": {
            "raw": res_bev,
            "mAP_Easy": map_easy_bev,
            "mAP_Mod": map_mod_bev,
            "mAP_Hard": map_hard_bev,
            "mAP9": map9_bev,
            "Car": car_bev,
            "Pedestrian": ped_bev,
            "Cyclist": cyc_bev,
        },
        "timing": timing,
        "params_m": spec.get("params_approx_m", 0.0),
        "type": spec.get("type", "3D Detector")
    }

    print(f"✓ {model_name}:")
    print(f"   - 3D  AP (R40): mAP Easy: {map_easy_3d:.2f}% | mAP Mod: {map_mod_3d:.2f}% | mAP Hard: {map_hard_3d:.2f}% | mAP9: {map9_3d:.2f}%")
    print(f"   - BEV AP (R40): mAP Easy: {map_easy_bev:.2f}% | mAP Mod: {map_mod_bev:.2f}% | mAP Hard: {map_hard_bev:.2f}% | mAP9: {map9_bev:.2f}%")
    print(f"   - Speed       : {timing['fps']:.2f} FPS ({timing['latency_ms']:.2f} ms/frame)")

print("=" * 80)
print("✓ Completed reference R40 evaluation for all models.")

In [ ]:
# ==============================================================================
# Cell 8: Comprehensive Comparative Benchmark Report & Publication Tables
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 1. BẢNG 1: 3D OBJECT DETECTION PERFORMANCE FULL (AP_3D R40 %)
# Khớp 100% các cột chuẩn: Model, Architecture, Protocol, Params, Latency ms, FPS,
# mAP Easy, mAP Mod, mAP Hard, mAP9, Car (E/M/H), Ped (E/M/H), Cyc (E/M/H)
# ------------------------------------------------------------------------------
rows_3d_full = []
for model_name, record in evaluation_records.items():
    data = record["3d"]
    t = record["timing"]
    car, ped, cyc = data["Car"], data["Pedestrian"], data["Cyclist"]
    
    rows_3d_full.append({
        "Model": model_name,
        "Architecture": record["type"],
        "AP Protocol": "Reference R40",
        "Params (M)": record["params_m"],
        "Latency (ms)": f"{t['latency_ms']:.2f}",
        "FPS": f"{t['fps']:.2f}",
        "mAP Easy (%)": f"{data['mAP_Easy']:.2f}",
        "mAP Moderate (%)": f"{data['mAP_Mod']:.2f}",
        "mAP Hard (%)": f"{data['mAP_Hard']:.2f}",
        "mAP9 (%)": f"{data['mAP9']:.2f}",
        "Car Easy (%)": f"{car['Easy']:.2f}",
        "Car Moderate (%)": f"{car['Moderate']:.2f}",
        "Car Hard (%)": f"{car['Hard']:.2f}",
        "Ped. Easy (%)": f"{ped['Easy']:.2f}",
        "Ped. Moderate (%)": f"{ped['Moderate']:.2f}",
        "Ped. Hard (%)": f"{ped['Hard']:.2f}",
        "Cyc. Easy (%)": f"{cyc['Easy']:.2f}",
        "Cyc. Moderate (%)": f"{cyc['Moderate']:.2f}",
        "Cyc. Hard (%)": f"{cyc['Hard']:.2f}",
    })

df_3d_full = pd.DataFrame(rows_3d_full)

# ------------------------------------------------------------------------------
# 2. BẢNG 2: BIRD'S EYE VIEW DETECTION PERFORMANCE FULL (AP_BEV R40 %)
# ------------------------------------------------------------------------------
rows_bev_full = []
for model_name, record in evaluation_records.items():
    data = record["bev"]
    t = record["timing"]
    car, ped, cyc = data["Car"], data["Pedestrian"], data["Cyclist"]
    
    rows_bev_full.append({
        "Model": model_name,
        "Architecture": record["type"],
        "AP Protocol": "Reference R40",
        "Params (M)": record["params_m"],
        "Latency (ms)": f"{t['latency_ms']:.2f}",
        "FPS": f"{t['fps']:.2f}",
        "mAP Easy (%)": f"{data['mAP_Easy']:.2f}",
        "mAP Moderate (%)": f"{data['mAP_Mod']:.2f}",
        "mAP Hard (%)": f"{data['mAP_Hard']:.2f}",
        "mAP9 (%)": f"{data['mAP9']:.2f}",
        "Car Easy (%)": f"{car['Easy']:.2f}",
        "Car Moderate (%)": f"{car['Moderate']:.2f}",
        "Car Hard (%)": f"{car['Hard']:.2f}",
        "Ped. Easy (%)": f"{ped['Easy']:.2f}",
        "Ped. Moderate (%)": f"{ped['Moderate']:.2f}",
        "Ped. Hard (%)": f"{ped['Hard']:.2f}",
        "Cyc. Easy (%)": f"{cyc['Easy']:.2f}",
        "Cyc. Moderate (%)": f"{cyc['Moderate']:.2f}",
        "Cyc. Hard (%)": f"{cyc['Hard']:.2f}",
    })

df_bev_full = pd.DataFrame(rows_bev_full)

# ------------------------------------------------------------------------------
# 3. BẢNG 3: BẢNG TỔNG HỢP NHANH (EXECUTIVE BENCHMARK SUMMARY)
# Phù hợp cho bảng chính tóm tắt trong bài báo
# ------------------------------------------------------------------------------
rows_summary = []
for model_name, record in evaluation_records.items():
    t = record["timing"]
    d3 = record["3d"]
    dbev = record["bev"]
    rows_summary.append({
        "Model": model_name,
        "Architecture": record["type"],
        "Params (M) ↓": record["params_m"],
        "Latency (ms) ↓": f"{t['latency_ms']:.2f}",
        "FPS ↑": f"{t['fps']:.2f}",
        "3D mAP Mod (%) ↑": f"{d3['mAP_Mod']:.2f}",
        "3D mAP9 (%) ↑": f"{d3['mAP9']:.2f}",
        "BEV mAP Mod (%) ↑": f"{dbev['mAP_Mod']:.2f}",
        "BEV mAP9 (%) ↑": f"{dbev['mAP9']:.2f}",
    })

df_summary = pd.DataFrame(rows_summary)

# Hiển thị trên Notebook
print("=" * 110)
print("📊 BẢNG 1: 3D OBJECT DETECTION PERFORMANCE FULL (AP_3D R40 % - IoU: Car 0.7, Ped 0.5, Cyc 0.5)")
print("=" * 110)
display(df_3d_full)

print("\n" + "=" * 110)
print("📊 BẢNG 2: BIRD'S EYE VIEW DETECTION PERFORMANCE FULL (AP_BEV R40 % - IoU: Car 0.7, Ped 0.5, Cyc 0.5)")
print("=" * 110)
display(df_bev_full)

print("\n" + "=" * 110)
print("📊 BẢNG 3: EXECUTIVE SUMMARY (PARAMS, SPEED, 3D & BEV mAP)")
print("=" * 110)
display(df_summary)

# ------------------------------------------------------------------------------
# 4. XUẤT CÁC FILE CSV & BÁO CÁO MARKDOWN
# ------------------------------------------------------------------------------
csv_3d_path = BENCHMARK_DIR / "benchmark_3D_mmdet3d_full.csv"
csv_bev_path = BENCHMARK_DIR / "benchmark_BEV_mmdet3d_full.csv"
csv_sum_path = BENCHMARK_DIR / "benchmark_summary_mmdet3d.csv"
md_report_path = BENCHMARK_DIR / "benchmark_tables_paper.md"

df_3d_full.to_csv(csv_3d_path, index=False)
df_bev_full.to_csv(csv_bev_path, index=False)
df_summary.to_csv(csv_sum_path, index=False)

with open(md_report_path, "w", encoding="utf-8") as f:
    f.write("# MMDetection3D Benchmark on KITTI (Reference R40 Protocol)\n\n")
    f.write(f"Evaluated on split `{VAL_SPLIT}` with **{len(val_ids):,} frames**.\n\n")
    f.write("### Table 1: Complete 3D Object Detection Performance (AP_3D R40 %)\n\n")
    f.write(df_3d_full.to_markdown(index=False) + "\n\n")
    f.write("### Table 2: Complete Bird's Eye View Detection Performance (AP_BEV R40 %)\n\n")
    f.write(df_bev_full.to_markdown(index=False) + "\n\n")
    f.write("### Table 3: Executive Summary (Efficiency & Performance)\n\n")
    f.write(df_summary.to_markdown(index=False) + "\n")

print(f"\n✓ Exported complete CSVs:")
print(f"   - 3D Full  : {csv_3d_path}")
print(f"   - BEV Full : {csv_bev_path}")
print(f"   - Summary  : {csv_sum_path}")
print(f"   - Markdown : {md_report_path}")

In [ ]:
# ==============================================================================
# Cell 9: Qualitative Comparison Inspector (Ground Truth vs Model Predictions)
# ==============================================================================
import random
from pathlib import Path

# Chọn một frame ID để kiểm tra dự đoán giữa các mô hình
SAMPLE_FRAME_ID = val_ids[0] if len(val_ids) > 0 else "000001"

print(f"Inspecting Frame ID: {SAMPLE_FRAME_ID}")
gt_path = RAW_TRAINING / f"label_2/{SAMPLE_FRAME_ID}.txt"
if gt_path.is_file():
    print("--- Ground Truth Labels ---")
    print(gt_path.read_text().strip())

for model_name in ACTIVE_MODELS:
    pred_file = PREDICTIONS_DIR / model_name / f"{SAMPLE_FRAME_ID}.txt"
    if not pred_file.is_file():
        candidates = list((PREDICTIONS_DIR / model_name).rglob(f"{SAMPLE_FRAME_ID}.txt"))
        if candidates:
            pred_file = candidates[0]
            
    if pred_file.is_file():
        content = pred_file.read_text().strip()
        lines = content.splitlines()
        print(f"\n--- {model_name} Detections ({len(lines)} objects) ---")
        for line in lines[:5]:
            print("  ", line)
        if len(lines) > 5:
            print(f"   ... and {len(lines) - 5} more detections.")